# TEST OF MY BYTE PAIR ENCODING TOKENIZER

This notebook contains examples of my BPE tokenizer in action. Its purpose is twofold:

- **Stress test** : See how my tokenizer behaves in extreme and uncommon cases.

- **How to use tutorial** : Show the reader how to use the tokenizer.

> **Note on Reliability**: The following tests include *edge cases* like control characters and nested special tokens to prove the robustness of the serialization system.

In [2]:
from xunko import Anakizer

## Normal use case of the tokenizer

We show a normal usage of the tokenizer. The steps we follow are:

- We initialize an empty tokenizer with the default parameters.

- We train the tokenizer using the most famous spanish book **El ingenioso hidalgo Don Quijote de la Mancha** with a vocabulary size of 200 tokens.

- We use the tokenizer on a series of spanish sentences from the book. These contain special characters and tokens to show how the tokenizer handles them.

- We save the tokenizer state and load it again.

- We check that the loading process leaves the tokenizer state intact.

> **Note on performance**: This is an educational project so readibility and the use of no external libraries were priorities while performance was not. 

In [3]:
tokenizer = Anakizer(unk = '<|unk|>', bos = '<|bos|>', eos = '<|eos|>')
print(f'Say hello to our tokenizer!\n{tokenizer}\n\n')

with open('quijote.txt', mode = 'rt', encoding = 'utf-8') as file:
    train_text = file.read()
tokenizer.train(text = train_text, vocab_size = 200)
print(f'Vocabulary of the tokenizer after training with \'El Quijote\': \n{tokenizer.token_to_id}\n\n')

sentences = [
    '<|bos|>En un lugar de la Mancha, de cuyo nombre no quiero acordarme.<|eos|>',
    '¡Qué locura es esta! ¿Acaso no veis que son gigantes y no molinos?',
    'La libertad, Sancho, es uno de los más preciosos dones que a los hombres dieron los cielos.\t'
]
for sentence in sentences: 
    tokenized_sentence = tokenizer.encode(sentence)
    detokenized_sentence = tokenizer.decode(tokenized_sentence, skip_specials = False)
    print(f'The sentence \'{sentence}\' tokenized looks like : \n{tokenized_sentence}')
    print(f'The reconstruction of the sentence by the tokenizer is : \n{detokenized_sentence}\n')
print('If you do not want to see the special characters present in the original sentences just use skip_specials = True\n\n')

(special_tokens, token_to_id, id_pairs_to_id) = (
    tokenizer.special_tokens.copy(),
    tokenizer.token_to_id.copy(), 
    tokenizer.id_pairs_to_id.copy()
)
tokenizer.save('quijote_save.txt')
tokenizer.load('quijote_save.txt')

print(f'Are the special tokens the same after loadig: {tokenizer.special_tokens == special_tokens}')
print(f'Are the vocabularies the same after loading: {tokenizer.token_to_id == token_to_id}')
print(f'Are the merging rules the same after loading: {tokenizer.id_pairs_to_id == id_pairs_to_id}')

Say hello to our tokenizer!
Untrained Anakizer with special tokens: <|unk|>, <|bos|>, <|eos|>


Vocabulary of the tokenizer after training with 'El Quijote': 
{'<|unk|>': 0, '<|bos|>': 1, '<|eos|>': 2, 'E': 3, 'l': 4, ' ': 5, 'i': 6, 'n': 7, 'g': 8, 'e': 9, 'o': 10, 's': 11, 'h': 12, 'd': 13, 'a': 14, 'Q': 15, 'u': 16, 'j': 17, 't': 18, 'M': 19, 'c': 20, '\n': 21, 'T': 22, 'A': 23, 'S': 24, 'Y': 25, ',': 26, 'J': 27, 'G': 28, 'r': 29, 'b': 30, 'C': 31, 'á': 32, 'm': 33, 'R': 34, 'y': 35, 'ñ': 36, 'q': 37, 'f': 38, 'v': 39, 'p': 40, 'é': 41, 'í': 42, ';': 43, '.': 44, 'V': 45, 'I': 46, 'O': 47, 'N': 48, 'D': 49, 'L': 50, 'ó': 51, 'U': 52, '1': 53, '6': 54, '0': 55, '4': 56, 'F': 57, 'P': 58, 'ú': 59, 'z': 60, ':': 61, 'B': 62, 'É': 63, 'x': 64, 'Ó': 65, '¿': 66, '?': 67, '-': 68, '¡': 69, '!': 70, 'X': 71, 'Z': 72, 'ü': 73, '»': 74, 'H': 75, "'": 76, 'Á': 77, 'Í': 78, 'Ñ': 79, '«': 80, '(': 81, ')': 82, '"': 83, 'ï': 84, 'Ú': 85, 'W': 86, ']': 87, 'à': 88, '7': 89, '5': 90, '2': 91, '3'

## Stress test 

We will now go very hard on the tokenizer to see how it reacts to very difficult and, in all honesty, uncommon cases. In particular we want to test:

- **Atomicity of the special tokens**: What happens if the training test contains special tokens. The tokenizer should always treat them as atomic units and thus it should not learn pieces of them.

- **Nested special tokens**: What happens if a special tokens is a substring of another special token. The tokenizer should treat both of them as atomic units and thus recognize the longest one without splitting it into the smaller special token and the rest of the characters.

- **Separator collision** What happens if the training test and the resulting vocabulary contain the tabulator and vertical tabulator special characters. These two are used as the separators of the save file so unless they are correctly escaped the loading process will fail.

- **Empty vocabularies** What happens if we work with an untrained tokenizer. The encode and decode methods should raise a RuntimeError warning the user and the saving and loading process should work normally keeping the tokenizer empty.

In [4]:
tokenizer = Anakizer('<|U\vNK|>', '<|B\tOS|>', '[EOS]', '<|SPECIAL', '<|SPECIAL_TOKEN|>')

with open('stress.txt', mode = 'rt', encoding = 'utf-8') as file:
    train_text = file.read()
tokenizer.train(text = train_text, vocab_size = 1000)
print(f'Vocabulary of the tokenizer after training with the stress dataset: \n{tokenizer.token_to_id}\n')

for pos in range(1, len('<|SPECIAL_TOKEN|>')):
    token_piece = '<|SPECIAL_TOKEN|>'[0 : pos]
    print(f'Did the tokenizer learn the {token_piece} substring of the \'<|SPECIAL_TOKEN|>\' special token: {token_piece in tokenizer.token_to_id}')
print('')
sentences = [
    r'La ruta del archivo es C:\\Users\\nicolas\\tokens y no C:\Users\nicolas\tokens. Recuerda que \\\\n no es lo mismo que \n ni que \\\n.',
    '<|SPECIAL_TOKEN|><|SPECIAL|><|SPECIAL_TOKEN|><|SPECIAL_TOKEN|>TOKEN|>',
    'El robot 🤖 dice que el café ☕ cuesta 5€ y que mañana lloverá en Cañitas de la Sierra... \v ¿o no? 🚀',
    'Cuidado con <|SPECIAL_TOKE (le falta la N) y con < |SPECIAL|> (tiene un espacio). No los confundas con <|SPECIAL_TOKEN|>.'
]
for sentence in sentences: 
    tokenized_sentence = tokenizer.encode(sentence)
    detokenized_sentence = tokenizer.decode(tokenized_sentence, skip_specials = False)
    print(f'The sentence \'{sentence}\' tokenized looks like : \n{tokenized_sentence}')
    print(f'The reconstruction of the sentence by the tokenizer is : \n{detokenized_sentence}\n')

(special_tokens, token_to_id, id_pairs_to_id) = (
    tokenizer.special_tokens.copy(),
    tokenizer.token_to_id.copy(), 
    tokenizer.id_pairs_to_id.copy()
)
tokenizer.save('stress_save.txt')
tokenizer.load('stress_save.txt')
print(f'\nAre the special characters \\t and \\v in the vocabularies: {'\t' in tokenizer.token_to_id and '\v' in tokenizer.token_to_id}')
print(f'Are they in the special tokens: {'\t' in ''.join(tokenizer.special_tokens) and '\v' in ''.join(tokenizer.special_tokens)}')
print(f'Are the special tokens the same after loadig: {tokenizer.special_tokens == special_tokens}')
print(f'Are the vocabularies the same after loading: {tokenizer.token_to_id == token_to_id}')
print(f'Are the merging rules the same after loading: {tokenizer.id_pairs_to_id == id_pairs_to_id}')

empty = Anakizer('<|unk|>', None, None)
try:
    empty.encode('Hello world')
except RuntimeError:
    print('\n\nThe encode method correctly warned us!')
try:
    empty.decode([0, 1, 4, 5])
except RuntimeError:
    print('The decode method correctly warned us!\n')

(special_tokens, token_to_id, id_pairs_to_id) = (
    empty.special_tokens.copy(),
    empty.token_to_id.copy(), 
    empty.id_pairs_to_id.copy()
)
empty.save('empty_save.txt')
empty.load('empty_save.txt')

print(f'Are the special tokens the same after loadig: {empty.special_tokens == special_tokens}')
print(f'Are the vocabularies the same after loading: {empty.token_to_id == token_to_id}')
print(f'Are the merging rules the same after loading: {empty.id_pairs_to_id == id_pairs_to_id}')

Vocabulary of the tokenizer after training with the stress dataset: 
{'<|U\x0bNK|>': 0, '<|B\tOS|>': 1, '[EOS]': 2, '<|SPECIAL': 3, '<|SPECIAL_TOKEN|>': 4, '-': 5, ' ': 6, 'S': 7, 'T': 8, 'A': 9, 'R': 10, 'O': 11, 'F': 12, 'U': 13, 'L': 14, 'I': 15, 'M': 16, 'E': 17, '\n': 18, 'u': 19, 'c': 20, 'a': 21, 's': 22, 'e': 23, 'p': 24, 'o': 25, 'g': 26, 'r': 27, 'f': 28, 'v': 29, 'i': 30, 't': 31, 'y': 32, 'ó': 33, ':': 34, '"': 35, '<': 36, '|': 37, 'B': 38, '\t': 39, '>': 40, '¡': 41, 'h': 42, 'd': 43, 'l': 44, 'n': 45, '!': 46, '.': 47, 'H': 48, 'b': 49, 'í': 50, 'm': 51, 'x': 52, 'G': 53, ',': 54, 'q': 55, 'k': 56, 'P': 57, 'C': 58, '_': 59, 'K': 60, 'N': 61, '(': 62, ')': 63, 'z': 64, '¿': 65, 'á': 66, '?': 67, 'ñ': 68, '[': 69, '\x0b': 70, ']': 71, "'": 72, 'V': 73, '\\': 74, 'ú': 75, 'Y': 76, 'X': 77, 'D': 78, 'Q': 79, '+': 80, '🍦': 81, '=': 82, '🚀': 83, '5': 84, '3': 85, '1': 86, '0': 87, '🤖': 88, 'j': 89, '🔥': 90, 'é': 91, ';': 92, 'Z': 93, 'o ': 94, 'e ': 95, 'a ': 96, 'l ': 97, 'n